# Using Jupyter Notebooks



In [18]:
# SPSS-style Logistic Regression Word Report Generator
# Works in Jupyter. Adjust file paths if needed.
import os
import math
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from docx import Document
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.shared import Pt

# ---------- CONFIG ----------
EXCEL_PATH = "/content/123.xlsx"     # <- your Excel file
OUTPUT_DOCX = "LR_Brand_Loyalty_SPSSstyle.docx1"
DEPENDENT = "Loyalty"
PREDICTORS = ["Attitude to Brand", "Attitude to Product", "Attitude to Shopping"]
CUT = 0.5   # classification cut value (like SPSS default)
GROUPS_HL = 10  # groups for Hosmer-Lemeshow
# ----------------------------

# Helper - pretty number
def r(n, d=4):
    try:
        return round(float(n), d)
    except:
        return n

# Read dataset
df = pd.read_excel(EXCEL_PATH)

# Case processing summary (non-missing)
case_total = len(df)
missing_any = df[[DEPENDENT] + PREDICTORS].isnull().any(axis=1).sum()
included = case_total - missing_any

# Ensure dependent encoding: map if strings
if df[DEPENDENT].dtype == object:
    # try mapping Loyal/Disloyal to 1/0
    mapping = {"Loyal": 1, "Disloyal": 0, "loyal": 1, "disloyal": 0}
    if set(df[DEPENDENT].dropna().unique()).intersection(set(mapping.keys())):
        df[DEPENDENT] = df[DEPENDENT].map(mapping)
# If still not numeric, try to coerce
df = df.dropna(subset=[DEPENDENT])  # drop missing dependent rows
df[DEPENDENT] = pd.to_numeric(df[DEPENDENT], errors='coerce')
df = df.dropna(subset=[DEPENDENT])
df = df.dropna(subset=PREDICTORS)  # require predictors present for final analysis

# Recompute counts
case_total = len(df) + missing_any
included = len(df)

# Fit null model and full model
X = df[PREDICTORS]
X_const = sm.add_constant(X, has_constant='add')
y = df[DEPENDENT].astype(int)

# Fit full logit
model_full = sm.Logit(y, X_const)
res_full = model_full.fit(disp=False)

# Fit null model (intercept-only model)
X0 = np.ones((len(y), 1))      # single column of 1s (intercept)
model_null = sm.Logit(y, X0)
res_null = model_null.fit(disp=False)

# Log-likelihoods and pseudo R2
llf = res_full.llf
llnull = res_null.llf
lr_stat = -2 * (llnull - llf)  # Omnibus LR chi2
df_model = len(PREDICTORS)
lr_pvalue = stats.chi2.sf(lr_stat, df_model)

n = len(y)
# Cox & Snell and Nagelkerke
cox_snell = 1 - np.exp((2 / n) * (llnull - llf))
nagelkerke = cox_snell / (1 - np.exp(2 * llnull / n))

# Hosmer-Lemeshow test implementation
pred_probs = res_full.predict(X_const)
df['pred_prob'] = pred_probs
df['pred_group'] = pd.qcut(df['pred_prob'], GROUPS_HL, duplicates='drop')  # quantile groups
# Build HL table
hl_table = []
# For each group compute observed and expected counts
groups = df.groupby('pred_group')
for name, group in groups:
    obs_loyal = int(group[DEPENDENT].sum())
    obs_total = len(group)
    exp_loyal = group['pred_prob'].sum()
    exp_disloyal = obs_total - exp_loyal
    obs_disloyal = obs_total - obs_loyal
    # accumulate components
    # HL chi-square component: (O - E)^2 / E for both categories if E>0
    comp = 0.0
    for O, E in [(obs_loyal, exp_loyal), (obs_disloyal, exp_disloyal)]:
        if E > 0:
            comp += (O - E) ** 2 / E
    hl_table.append({
        'group': str(name),
        'obs_loyal': obs_loyal,
        'exp_loyal': r(exp_loyal,3),
        'obs_disloyal': obs_disloyal,
        'exp_disloyal': r(exp_disloyal,3),
        'total': obs_total,
        'comp': comp
    })
hl_chi2 = sum([g['comp'] for g in hl_table])
hl_df = len(hl_table) - 2
hl_p = stats.chi2.sf(hl_chi2, hl_df) if hl_df > 0 else np.nan

# Classification table
df['pred_class'] = (df['pred_prob'] >= CUT).astype(int)
tp = len(df[(df[DEPENDENT] == 1) & (df['pred_class'] == 1)])
tn = len(df[(df[DEPENDENT] == 0) & (df['pred_class'] == 0)])
fp = len(df[(df[DEPENDENT] == 0) & (df['pred_class'] == 1)])
fn = len(df[(df[DEPENDENT] == 1) & (df['pred_class'] == 0)])
accuracy = (tp + tn) / len(df) * 100

# Variables in the equation (coeffs summary)
coef_table = []
params = res_full.params
bse = res_full.bse
# statsmodels Logit returns z-value and p-value
zvalues = res_full.tvalues
pvalues = res_full.pvalues
conf = res_full.conf_int()
for name in params.index:
    coef_table.append({
        'name': name,
        'B': r(params[name], 4),
        'S.E.': r(bse[name], 4),
        'z/Wald': r(zvalues[name], 4),
        'Sig.': r(pvalues[name], 4),
        'CI_low': r(conf.loc[name,0], 4),
        'CI_high': r(conf.loc[name,1], 4),
        'Exp(B)': r(np.exp(params[name]), 4)
    })

# Variables not in the equation (Block 0 style) - compute univariate LR statistic for each predictor vs null
not_in_eq = []
for var in PREDICTORS:
    X_var = sm.add_constant(df[[var]], has_constant='add')
    try:
        res_u = sm.Logit(y, X_var).fit(disp=False)
        ll_u = res_u.llf
        chi2 = -2 * (llnull - ll_u)
        p = stats.chi2.sf(chi2, 1)
    except Exception as e:
        chi2 = np.nan
        p = np.nan
    not_in_eq.append({'var': var, 'ScoreChi2_like': r(chi2,4), 'df':1, 'Sig': r(p,4)})

# Variables not in the equation: we can format as SPSS shows (score-like values)
overall_chi2 = lr_stat
overall_p = lr_pvalue

# ASCII predicted prob plot (SPSS-like)
# We'll produce a line of symbols where each symbol represents 0.25 cases (approx)
sorted_df = df.sort_values('pred_prob').reset_index(drop=True)
symbols = []
for _, row in sorted_df.iterrows():
    symbols.append('L' if row[DEPENDENT]==1 else 'D')
# Create groups of 4 cases to represent each symbol block (to mimic SPSS's condensed plot)
# But simpler: place sequence of 'D' and 'L' with thin spacing and a probability scale below
plot_symbols = ''.join(symbols)

# -------------------------
# Build Word Document
# -------------------------
doc = Document()
style = doc.styles['Normal']
font = style.font
font.name = 'Arial'
font.size = Pt(10)

def add_heading(text, level=1):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.bold = True
    run.font.size = Pt(11 if level==1 else 10)

# Title
add_heading("Logistic Regression", level=1)
doc.add_paragraph("")  # blank line

# --- Notes (mirroring SPSS)
doc.add_paragraph("Notes")
doc.add_paragraph(f"Data file: {EXCEL_PATH}")
doc.add_paragraph(f"Number of rows in working data file (cases): {len(df)}")
doc.add_paragraph("")

# Case Processing Summary
add_heading("Case Processing Summary", level=2)
t = doc.add_table(rows=1, cols=3)
hdr = t.rows[0].cells
hdr[0].text = "N"
hdr[1].text = "Percent"
hdr[2].text = "Notes"
row = t.add_row().cells
row[0].text = str(included)
row[1].text = f"{r(included / case_total * 100,2)}%"
row[2].text = "Included in analysis"
row = t.add_row().cells
row[0].text = str(missing_any)
row[1].text = f"{r(missing_any / case_total * 100,2)}%"
row[2].text = "Missing cases"

doc.add_paragraph("")

# Dependent Variable Encoding
add_heading("Dependent Variable Encoding", level=2)
t = doc.add_table(rows=1, cols=2)
hdr = t.rows[0].cells
hdr[0].text = "Original Value"
hdr[1].text = "Internal Value"
# attempt to detect labels
unique_vals = sorted(df[DEPENDENT].unique())
# assume internal 0/1
for val in unique_vals:
    row = t.add_row().cells
    row[0].text = str(val)
    row[1].text = str(int(val))

doc.add_paragraph("")

# Block 0 - Beginning Block
add_heading("Block 0: Beginning Block", level=2)
doc.add_paragraph("Classification Table")
t = doc.add_table(rows=1, cols=4)
hdr = t.rows[0].cells
hdr[0].text = "Observed"
hdr[1].text = "Predicted: Disloyal"
hdr[2].text = "Predicted: Loyal"
hdr[3].text = "Percentage Correct"

# Compute observed counts for null model (predict everything as majority class)
maj_class = int(y.mode()[0])
obs_disloyal = sum(y==0)
obs_loyal = sum(y==1)
# If majority is 0, predicted all as 0 -> percent correct = obs_disloyal/total
if maj_class == 0:
    pct_correct_disloyal = r(obs_disloyal / len(y) * 100,2)
    pct_correct_loyal = 0.0
else:
    pct_correct_loyal = r(obs_loyal / len(y) * 100,2)
    pct_correct_disloyal = 0.0

row = t.add_row().cells
row[0].text = "Disloyal"
row[1].text = str(obs_disloyal if maj_class==0 else 0)
row[2].text = str(0 if maj_class==0 else obs_disloyal)
row[3].text = f"{pct_correct_disloyal}%"

row = t.add_row().cells
row[0].text = "Loyal"
row[1].text = str(0 if maj_class==0 else obs_loyal)
row[2].text = str(obs_loyal if maj_class==1 else 0)
row[3].text = f"{pct_correct_loyal}%"

doc.add_paragraph("Overall Percentage: " + str(r(max(pct_correct_disloyal, pct_correct_loyal),2)) + "%")
doc.add_paragraph("")

# Variables not in the equation (Block 0 style)
add_heading("Variables not in the Equation", level=2)
t = doc.add_table(rows=1, cols=4)
hdr = t.rows[0].cells
hdr[0].text = "Variables"
hdr[1].text = "Score"
hdr[2].text = "df"
hdr[3].text = "Sig."
for ni in not_in_eq:
    row = t.add_row().cells
    row[0].text = ni['var']
    row[1].text = str(ni['ScoreChi2_like'])
    row[2].text = str(ni['df'])
    row[3].text = str(ni['Sig'])
doc.add_paragraph("")

# Omnibus Tests of Model Coefficients
add_heading("Omnibus Tests of Model Coefficients", level=2)
t = doc.add_table(rows=1, cols=3)
hdr = t.rows[0].cells
hdr[0].text = "Chi-square"
hdr[1].text = "df"
hdr[2].text = "Sig."
row = t.add_row().cells
row[0].text = str(r(overall_chi2,4))
row[1].text = str(int(df_model))
row[2].text = str(r(overall_p, 6))
doc.add_paragraph("")

# Model Summary
add_heading("Model Summary", level=2)
t = doc.add_table(rows=1, cols=4)
hdr = t.rows[0].cells
hdr[0].text = "-2 Log likelihood"
hdr[1].text = "Cox & Snell R Square"
hdr[2].text = "Nagelkerke R Square"
hdr[3].text = "AIC"
row = t.add_row().cells
row[0].text = str(r(-2 * llf,4))
row[1].text = str(r(cox_snell,4))
row[2].text = str(r(nagelkerke,4))
row[3].text = str(r(res_full.aic,4))
doc.add_paragraph("")

# Hosmer and Lemeshow Test
add_heading("Hosmer and Lemeshow Test", level=2)
t = doc.add_table(rows=1, cols=3)
hdr = t.rows[0].cells
hdr[0].text = "Chi-square"
hdr[1].text = "df"
hdr[2].text = "Sig."
row = t.add_row().cells
row[0].text = str(r(hl_chi2,4))
row[1].text = str(int(hl_df) if not math.isnan(hl_df) else "NA")
row[2].text = str(r(hl_p,4))
doc.add_paragraph("")

# Contingency table for HL test (group rows)
add_heading("Contingency Table for Hosmer and Lemeshow Test", level=2)
t = doc.add_table(rows=1, cols=5)
hdr = t.rows[0].cells
hdr[0].text = "Group"
hdr[1].text = "Observed Disloyal"
hdr[2].text = "Expected Disloyal"
hdr[3].text = "Observed Loyal"
hdr[4].text = "Expected Loyal"
for g in hl_table:
    row = t.add_row().cells
    row[0].text = g['group']
    row[1].text = str(g['obs_disloyal'])
    row[2].text = str(g['exp_disloyal'])
    row[3].text = str(g['obs_loyal'])
    row[4].text = str(g['exp_loyal'])
doc.add_paragraph("")

# Classification Table
add_heading("Classification Table", level=2)
t = doc.add_table(rows=1, cols=4)
hdr = t.rows[0].cells
hdr[0].text = "Observed"
hdr[1].text = "Predicted: Disloyal"
hdr[2].text = "Predicted: Loyal"
hdr[3].text = "Percentage Correct"
# fill table rows
row = t.add_row().cells
row[0].text = "Disloyal"
row[1].text = str(tn)
row[2].text = str(fp)
row[3].text = str(r(tn / (tn+fp) * 100,2) if (tn+fp)>0 else 0) + "%"
row = t.add_row().cells
row[0].text = "Loyal"
row[1].text = str(fn)
row[2].text = str(tp)
row[3].text = str(r(tp / (tp+fn) * 100,2) if (tp+fn)>0 else 0) + "%"
doc.add_paragraph("Overall Percentage: " + str(r(accuracy,2)) + "%")
doc.add_paragraph("Note: The cut value is " + str(CUT))
doc.add_paragraph("")

# Variables in the Equation
add_heading("Variables in the Equation", level=2)
cols = ["B","S.E.","z/Wald","Sig.","[0.025","0.975]","Exp(B)"]
t = doc.add_table(rows=1, cols=len(cols)+1)
hdr = t.rows[0].cells
hdr[0].text = ""
for i,col in enumerate(cols):
    hdr[i+1].text = col

for rowdict in coef_table:
    row = t.add_row().cells
    row[0].text = rowdict['name']
    row[1].text = str(rowdict['B'])
    row[2].text = str(rowdict['S.E.'])
    row[3].text = str(rowdict['z/Wald'])
    row[4].text = str(rowdict['Sig.'])
    row[5].text = str(rowdict['CI_low'])
    row[6].text = str(rowdict['CI_high'])
    # append Exp(B) in the last cell (we need to expand table columns if required)
    # For safety, ensure number of columns equals expected
    # If table has only len(cols)+1 cells, append Exp(B) to last cell
    row[-1].text = str(rowdict['Exp(B)'])

doc.add_paragraph("")

# Observed Groups and Predicted Probabilities ASCII-style plot
add_heading("Observed Groups and Predicted Probabilities", level=2)
doc.add_paragraph("Predicted Probability is of Membership for Loyal")
doc.add_paragraph("The Cut Value is " + str(CUT))
# Put simple symbol line (D/L)
p = doc.add_paragraph()
p.add_run(plot_symbols)
p.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT
doc.add_paragraph("Symbols: D - Disloyal ; L - Loyal")
doc.add_paragraph("Each symbol represents single case (ordered by predicted probability).")

# Save document
# Ensure output folder exists
out_dir = os.path.dirname(OUTPUT_DOCX)
if out_dir and (not os.path.exists(out_dir)):
    os.makedirs(out_dir, exist_ok=True)

doc.save(OUTPUT_DOCX)
print("SPSS-style report saved to:", OUTPUT_DOCX)


SPSS-style report saved to: LR_Brand_Loyalty_SPSSstyle.docx1


/tmp/ipython-input-2144500738.py:86: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = df.groupby('pred_group')


##Introduction
Jupyter notebooks is an open-source web-based Python editor which runs in your browser. It allows a combination of text written in a html-like format known as "markdown", such as the block of text you're reading right now, and inline code, tools and outputs such as this one:

In [ ]:
print("Hello World")

This combination allows for the procution of beautiful documents containing software, documentation and discussion. For larger codes you may wish to use Python in a stand-alone environment such as a traditional IDE. But for demonstration purposes Jupyter is a very useful tool.

Notebook files have the extension ".ipynb" extension. A Jupyter notebook is one of many environments you may run Python code.  Colab and the Jupyter notebook editor in Anaconda are two of the many pieces of software you may use to write and run a Jupyter notebook. For this course we recommend using the online Google Colab tool, but you can use Anaconda to run the notebooks on your own machine within an internet connection. On college computers, Jupyter can be used by launchng Anaconda from the Software Hub Apps Anywhere interface.

Note that exact interfaces will differ between different environments but the same functionality should be found in most environments. This course will be using the Colab environment.

## Cells and Executing Code

A notebooks is made up of one or more "cells". Cells can contain the html-like text used to generate text or code to be run by the user. A cell containing a piece of code may be recognised by the the  ```[]```  to the left of it. Code in these blocks can be run in a nubmer of ways. The simplest is click on the ```[ ]``` . This will execute the code. Try this with the code snippet below:

In [ ]:
print("Yes, it worked!")

You should have seen the message "Yes, it worked!" appear immediately beneath the code. This is the output of the code, which has been printed to the screen. You may also have noticed a number appear between the square brackets to the left of the code snippet. This indicates the order in which the code snippet has been executed. Code cells may be executed in any order and variables will be saved between execution of code snippets. To try this, execute the three codes snippets below in the following order:
- 1
- 2
- 3
- 2

In [ ]:
a="Message 1"

In [ ]:
print(a)

In [ ]:
a="Message 2"

The first time you ran code snippet 1 you should have seen "Message 1" as the output and the second time the output should have been "Message 2". This is because the first time it was run, the value assigned to the variable named "a" was "Message" as set by the first code snippet and the second time it was "Message 2" as set by the third code snippet. Note also the current numbers contained within square brackets. These help you to kno which cells have been executed and in which order.

##Sharing Jupyter Notebooks on Colab
When a Jupyter Notebook is shared with you on Colab, you will often receive access to the notebook which will alow you to run code, but not edit it. This should be the case for the notebooks that form part of this course. In this case you can select "Save a Copy in Drive" from the "File" menu to create a new copy that is yours and yo can edit.

For this course, it is reccommended that you create two copies. One of these should be the original copy without your edits, and another which you can edit to compelte exercises or expierment.

## Basic Jupyter Commands

Jupyter contains a number of useful tools for executing these cells. By using the "Runtime" menu, you can run multiple cells at a time using "Run all", "Run before", "Run selected" and "Run after".

You can clear output (this is the term for what is written under a code cell when it's executed) by clicking on the symbol to the left of it. You can clear all outputs from the notebook using the "Clear All Outputs" command on the "Edit" menu. Clearing the output will not unset variables set by the code snippets run, only remove the output printed to the screen.

To unset variables, use the "Restart Runtime" or "Reset Runtime" option in the Runtime menu. The "Interrupt Execution" command on the kernel menu will halt the procesing of code, which can be useful if you've accidentally written a piece of code that will never finish executing or if the code is taking too long to execute.

The "insert" menu allows you to create new cells. The "cell type" option in the "cell" menu allows you toggle the current cell type between the different cell types available:
- **Code**: Code snippets
- **Text**: The html-like language used to generate text, tables, equations, etc.

Alternatively, you can hover your mouse in the space after a cell and add a code or text cell there.

###Exercise

Try each of these commands from the different menus for yourself on this  notebook and ensure they behave as you would expect.

## Text Cells in Jupyter
You can include all sort so information in Jupyter text cells to obtain different effects. To see how each of the following examples is generated, double click on this cell. To return to the formatted text, run the cell.

### Headings
Headings can be generated using the hash symbol "#". The more of these there are, the smaller the heading. The sub-sub-heading above is an example.

### Tables
Tables can be created in a way similar to basic html, using the a comabination of the "|" and "-" symbols:

| This | is    |
|------|-------|
|   a  |  table|
| It's | fancy |

### Equations
Equations can be written in a way similar to LaTeX by surrouding the text with "\$" symbols:

$a=\frac{\int\limits_{0}^{\pi} \sin{(bx)} \textrm{d}x}{4}$

Don't worry if you don't understand the exact syntax used to generate this example. In your example of it in your exercise, try writing something very simple instead. If it looks like a simple algebraic expression, it will probably render how you intend.

### Code Snippets
You can write snippets of code in a text cell and they will be highlighted as if they were code written in a code cell. This can be useful for demonstrating a code feature in a textual way. For example:

```python
print ("Hello World")
```

There is not a way to run this code, it is merely normal text highlighted to look like code. The "python" which precedes the code itself tells Jupyter which language you are writing the code snippet in so it can be highlighted accorindly.

In some environments, text cells may also be referred to as "markdown" cells.

###Exercise
Try creating simple versions of each of the constructs above in a new text cell below this one.